# 07. GCNO-LR: Solid-path rejection ablation

**Purpose:** isolate whether rejecting local stencil sampling paths that intersect the solid airfoil mask improves prediction accuracy, independently of the existence of the local refinement branch.

This is a **post hoc development-validation control**, not an amendment to the earlier frozen official-test protocol. The original six notebooks, previously published weights, and official-test outputs are never modified or re-evaluated here.

**Configurations:** `masked` (selected Cartesian GCNO-LR) and `no_path_rejection` (same model without the solid-path validity check). Both retain the same encoder, spectral blocks, FiLM, adaptive stencil weights, two shared local updates, fluid-masked residual update, and output head. Out-of-domain endpoints remain disallowed in both configurations.

**Procedure:** three seeds (42, 123, 2026), 10,000 optimizer steps per seed/configuration, batch size 10, Adam 4e-4, betas (0.5, 0.999), existing split seed 31415, validation every 500 steps, best validation scaled-L1 checkpoint, exploratory paired validation uncertainty across seeds and airfoil identities. No official-test predictions are loaded.

**Required from notebooks 01 to 05:**

- `prepared/metadata.json`, `prepared/manifest.csv`, `prepared/splits/official_random_31415.json`, and corresponding prepared `.npz` sample files.
- `airfoil_publication_core.py` generated by Notebook 02.
- Versioned `airfoil_next_stage.py` helper produced by Notebooks 03 to 05.
- Recorded `runs/study_03/ablations_10k_3seeds/no_alignment_seed{seed}/run_identity.json` files, used to establish compatibility with the frozen selected model.

**Research question:** Does allowing local stencil paths to cross the airfoil solid mask affect near-airfoil errors, global field error, or computational cost under otherwise matched training conditions?

## 1. Execution controls

Default action is `status`, which **does not train**. Select `train` only after the checks pass. One execution advances at most one model/seed by a specified step chunk. Rerun the notebook after a Colab restart to continue. `summarize` performs analysis only after all six new runs are finished.

The prepared data and original core are **read-only**. New files are written only to `runs/study_06_mask_path_ablation_v1`.

In [ ]:
import os, sys, gc, json, hashlib, importlib.util, inspect, textwrap, types, copy
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Choose one: status | train | summarize
ACTION = os.environ.get("GCNO_MASK_ACTION", "status")
assert ACTION in {"status", "train", "summarize"}

PROJECT_ROOT = Path(os.environ.get(
    "AIRFOIL_PROJECT_ROOT",
    "/content/drive/MyDrive/Colab Notebooks/Deep-Flow-Prediction/"
    "rans_airfoil_project/publication_v2"
))
PREPARED_ROOT = PROJECT_ROOT / "prepared"
SPLIT_PATH = PREPARED_ROOT / "splits" / "official_random_31415.json"
CORE_PATH = PROJECT_ROOT / "airfoil_publication_core.py"
ABLATION_ROOT = PROJECT_ROOT / "runs" / "study_03" / "ablations_10k_3seeds"
STUDY_ROOT = PROJECT_ROOT / "runs" / "study_06_mask_path_ablation_v1"
REPORT_ROOT = STUDY_ROOT / "reports"

# Identical version of the validated helper from Notebooks 03-06.
HELPER_SHA256 = "8465a09fd40898a682a5339cacc40ba4bca82a2203873297376b197ad9668401"
HELPER_PATH = PROJECT_ROOT / "analysis_tools" / HELPER_SHA256[:12] / "airfoil_next_stage.py"
SEEDS = [42, 123, 2026]
VARIANTS = ["masked", "no_path_rejection"]
MAX_STEPS = 10_000
VALIDATION_EVERY = 500
CHUNK_STEPS = 1_000  # Multiple of 500; valid alternatives: 500, 1500, 3000
SESSION_MINUTES = 12  # Graceful stop at validation boundaries
BOOTSTRAP_REPETITIONS = 4_000
BOOTSTRAP_SEED = 812
MAKE_FIGURES = True
RUN_INFERENCE_TIMING = False

if ACTION == "train" and (CHUNK_STEPS < 500 or CHUNK_STEPS % VALIDATION_EVERY):
    raise ValueError("CHUNK_STEPS must be a positive multiple of VALIDATION_EVERY.")

if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

for required in [PREPARED_ROOT / "metadata.json", PREPARED_ROOT / "manifest.csv",
                 SPLIT_PATH, CORE_PATH, HELPER_PATH]:
    if not required.is_file():
        raise FileNotFoundError(f"Missing required file: {required}")
for seed in SEEDS:
    required = ABLATION_ROOT / f"no_alignment_seed{seed}" / "run_identity.json"
    if not required.is_file():
        raise FileNotFoundError(f"Existing selected-model identity not found: {required}")

STUDY_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
print("Action:", ACTION)
print("Development data:", PREPARED_ROOT)
print("New study output:", STUDY_ROOT)
print("Official-test folder is neither read nor modified.")

## 2. Load verified original code and split identities

The original core is not overwritten. Before importing it, verify its fingerprint against the completed `no_alignment` study. The versioned helper is also checked by SHA-256. The train/validation partition is reused exactly. These checks intentionally fail on incompatible versions.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(2**20), b""):
            h.update(part)
    return h.hexdigest()

if sha256_file(HELPER_PATH) != HELPER_SHA256:
    raise ValueError("Original experiment helper hash changed; restore the historical version.")

spec = importlib.util.spec_from_file_location("gcno_next_stage_for_mask_ablation", HELPER_PATH)
stage = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = stage
spec.loader.exec_module(stage)

original_identities = {
    seed: stage.read_json(ABLATION_ROOT / f"no_alignment_seed{seed}" / "run_identity.json")
    for seed in SEEDS
}
source_fingerprints = {v["code_sha256"] for v in original_identities.values()}
if len(source_fingerprints) != 1:
    raise ValueError("Original selected-model seeds use different architecture fingerprints.")
original_fp = next(iter(source_fingerprints))
core = stage.load_core(CORE_PATH, original_fp)
meta = core.verify_prepared_dataset(PREPARED_ROOT)
split = stage.read_json(SPLIT_PATH)
if meta["settings"].get("synthetic"):
    raise ValueError("Publication ablations require real CFD data, not synthetic fixtures.")
if meta["dataset_id"] != split["dataset_id"]:
    raise ValueError("Dataset identity does not match split identity.")
if stage.json_hash({k:v for k,v in split.items() if k!="split_id"}) != split["split_id"]:
    raise ValueError("Split manifest hash does not match its contents.")
if set(split["train_indices"]) & set(split["val_indices"]):
    raise ValueError("Training/validation sample overlap detected.")
source_configs = [original_identities[seed]["config"] for seed in SEEDS]
if not all(x == source_configs[0] for x in source_configs[1:]):
    raise ValueError("Selected-model source configurations differ across seeds.")
BASE_CONFIG = copy.deepcopy(source_configs[0])
for name, expected in {
    "model": "proposed", "aligned": False,
    "max_steps": MAX_STEPS, "batch_size": 10,
    "validation_every": VALIDATION_EVERY, "selection_metric": "legacy_l1"
}.items():
    if BASE_CONFIG.get(name) != expected:
        raise ValueError(f"Original configuration mismatch for {name}: {BASE_CONFIG.get(name)!r}")
if BASE_CONFIG.get("use_refiner", True) is not True:
    raise ValueError("Selected model unexpectedly disables local refinement.")
if BASE_CONFIG.get("adaptive", True) is not True:
    raise ValueError("Selected model unexpectedly disables adaptive stencil weighting.")
if BASE_CONFIG.get("loss", {}) != {}:
    raise ValueError("Selected model unexpectedly uses additional loss terms.")

# In addition to the sample-disjoint split, identify geometry overlap openly.
manifest = pd.read_csv(PREPARED_ROOT / "manifest.csv", dtype={"sample_id":str,"airfoil_id":str})
train_manifest = manifest[manifest.split == "train"].reset_index(drop=True)
train_ids = set(train_manifest.iloc[split["train_indices"]].airfoil_id)
val_ids = set(train_manifest.iloc[split["val_indices"]].airfoil_id)
print("Prepared data ID:", meta["dataset_id"])
print("Split ID:", split["split_id"])
print("Original code fingerprint:", original_fp)
print("Train samples:", len(split["train_indices"]), "Validation samples:", len(split["val_indices"]))
print("Geometry identities shared by train and validation:", len(train_ids & val_ids))
print("This validation analysis is exploratory; geometry-level overlap is disclosed.")

## 3. Create the versioned, one-operation model variant

The **original** `AlignedStencilRefiner.forward` function is obtained from the verified original core. A versioned module is constructed by replacing **only** the validity-checking block between two strict code anchors. Everything else, including feature differences, gates, softmax normalization, convolutional mixing, residual update and fluid correction mask, is preserved byte-for-byte from that function's source.

In the unmasked variant, sampled endpoints must remain inside the computational domain but may pass through solid-mask pixels. In the original masked variant, endpoint and intermediate solid paths are rejected. Both variants retain final multiplication of learned corrections by the fluid mask. No additional trainable parameters are introduced.

In [ ]:
orig_forward = textwrap.dedent(inspect.getsource(core.AlignedStencilRefiner.forward))
start_marker = "                # Reject samples whose bilinear support crosses the solid/domain boundary."
end_marker = "                sampled = F.grid_sample(work, sample_grid,"
if orig_forward.count(start_marker) != 1 or orig_forward.count(end_marker) != 1:
    raise ValueError("Original refiner implementation changed; manual review required.")
start = orig_forward.index(start_marker)
end = orig_forward.index(end_marker)
assert start < end
old_validity = orig_forward[start:end]
for token in ["grid_sample(b['fluid']", "for fraction", "valid = valid &"]:
    if token not in old_validity:
        raise ValueError(f"Unexpected original mask-validity code: missing {token}")

new_validity = """                # Ablation: keep domain support but permit crossing the solid airfoil mask.
                # Intermediate path points are inside the domain whenever both endpoints
                # are inside, because all stencil offsets are straight line segments.
                valid = ((sample_grid[..., 0] >= -1.0) &
                         (sample_grid[..., 0] <=  1.0) &
                         (sample_grid[..., 1] >= -1.0) &
                         (sample_grid[..., 1] <=  1.0)).unsqueeze(1)
"""
patched_forward = orig_forward[:start] + new_validity + orig_forward[end:]
if patched_forward.count("correction = self.film(correction, b['condition']) * b['fluid']") != 1:
    raise ValueError("Fluid-pixel update mask was unexpectedly removed.")
patched_forward = patched_forward.replace(
    "def forward(self, x, b):", "def unmasked_forward(self, x, b):", 1
)
if "def unmasked_forward(" not in patched_forward:
    raise AssertionError("Invalid patched function name.")
version_source = (
    '"""GCNO-LR mask-only development ablation, generated from verified historical refiner."""\n'
    'import numpy as np\nimport torch\nfrom torch.nn import functional as F\n\n'
).replace('\n','\n') + patched_forward
# The generated module's content address protects resume compatibility.
model_hash = hashlib.sha256(version_source.encode("utf-8")).hexdigest()
version_dir = STUDY_ROOT / "code" / model_hash[:12]
version_dir.mkdir(parents=True, exist_ok=True)
VERSION_FILE = version_dir / "mask_path_ablation_model.py"
if VERSION_FILE.exists():
    if VERSION_FILE.read_text(encoding="utf-8") != version_source:
        raise ValueError("Versioned model source differs from existing locked source.")
else:
    VERSION_FILE.write_text(version_source, encoding="utf-8")

spec_v = importlib.util.spec_from_file_location("gcno_mask_ablation_" + model_hash[:12], VERSION_FILE)
variant_module = importlib.util.module_from_spec(spec_v)
sys.modules[spec_v.name] = variant_module
spec_v.loader.exec_module(variant_module)

class MaskAblationCore:
    """Delegates all preprocessing/training methods to the verified original core."""
    def __init__(self, original, variant, source_hash):
        self._original = original
        self._variant = variant
        self._fp = hashlib.sha256(
            (original.implementation_fingerprint() + ":" + source_hash).encode()
        ).hexdigest()
    def implementation_fingerprint(self):
        return self._fp
    def model_factory(self, config):
        if config.get("model") != "proposed" or config.get("use_refiner", True) is not True:
            raise ValueError("This study permits only the two local-refinement variants.")
        if config.get("aligned") is not False or config.get("adaptive", True) is not True:
            raise ValueError("Both ablation variants must use adaptive Cartesian stencils.")
        flag = config.get("mask_path_rejection")
        if not isinstance(flag, bool):
            raise ValueError("mask_path_rejection must explicitly be true or false.")
        model = self._original.model_factory(config)
        if not flag:
            # Forward-method replacement does not construct weights or change RNG state.
            model.refiner.forward = types.MethodType(self._variant.unmasked_forward, model.refiner)
        return model
    def __getattr__(self, name):
        return getattr(self._original, name)

ablation_core = MaskAblationCore(core, variant_module, model_hash)
print("Versioned ablation source:", VERSION_FILE)
print("New source SHA256:", model_hash)
print("Composite training fingerprint:", ablation_core.implementation_fingerprint())

## 4. Lock matched configurations and run correctness tests

**Mandatory gates before training:** same trainable parameter names/count; identical initial weights for paired seeds; masked variant exactly matches the original model; domain validity rejects endpoints outside the crop; the unmasked variant permits a path over solid pixels; both models produce finite forward outputs and finite gradients. These tests use development data only.

In [ ]:
configs = {}
for name, reject in [("masked", True), ("no_path_rejection", False)]:
    cfg = copy.deepcopy(BASE_CONFIG)
    cfg["profile"] = "mask_path_development"
    cfg["mask_path_rejection"] = reject
    configs[name] = cfg

assert {k:v for k,v in configs["masked"].items() if k!="mask_path_rejection"} == {
    k:v for k,v in configs["no_path_rejection"].items() if k!="mask_path_rejection"
}

core.set_reproducibility(SEEDS[0])
original_model = core.model_factory(BASE_CONFIG)
core.set_reproducibility(SEEDS[0])
masked_model = ablation_core.model_factory(configs["masked"])
core.set_reproducibility(SEEDS[0])
unmasked_model = ablation_core.model_factory(configs["no_path_rejection"])

s0,s1,s2 = [m.state_dict() for m in [original_model, masked_model, unmasked_model]]
assert list(s0) == list(s1) == list(s2)
assert all(torch.equal(s0[k], s1[k]) and torch.equal(s0[k],s2[k]) for k in s0)
param_counts = [core.model_size(m)["real_scalar_parameters"] for m in
                [original_model, masked_model, unmasked_model]]
assert len(set(param_counts)) == 1
assert param_counts[0] == 5_690_558, f"Unexpected selected GCNO-LR parameter count: {param_counts}"

# Take one real sample from the TRAIN partition. Never open official test targets.
stats = core.fit_condition_stats(PREPARED_ROOT, split)
small_set = core.AirfoilDataset(
    PREPARED_ROOT, "train", split["train_indices"][:1], stats, configs["masked"]
)
item = small_set[0]
batch = {k: v.unsqueeze(0) if torch.is_tensor(v) else [v] for k,v in item.items()}

test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for model in (original_model, masked_model, unmasked_model):
    model.to(test_device).eval()
batch = core.to_device(batch, test_device)
with torch.inference_mode():
    result_original = original_model(batch)["prediction"]
    result_masked = masked_model(batch)["prediction"]
    result_unmasked = unmasked_model(batch)["prediction"]
assert torch.equal(result_original, result_masked), "Masked model differs from original source."
assert result_original.shape == result_unmasked.shape == (1,3,128,128)
assert torch.isfinite(result_unmasked).all()

# Directly exercise the trained-parameter-equivalent refiner on synthetic geometry:
# a stencil of radius 3 may cross a one-pixel obstacle.
torch.manual_seed(123)
latent = torch.randn(1, configs["masked"].get("base",32), 9, 9,device=test_device)
obstacle_fluid = torch.ones(1,1,9,9,device=test_device)
obstacle_fluid[:, :, 4, 4] = 0
probe = {
    "fluid":obstacle_fluid,
    "direction":torch.tensor([[1.,0.]],device=test_device),
    "normals":torch.zeros(1,2,9,9,device=test_device),
    "condition":torch.zeros(1,3,device=test_device),
    "features":torch.zeros(1,12,9,9,device=test_device)
}
with torch.inference_mode():
    ref_masked = masked_model.refiner(latent,probe)
    ref_unmasked = unmasked_model.refiner(latent,probe)
assert torch.isfinite(ref_masked).all() and torch.isfinite(ref_unmasked).all()
assert not torch.allclose(ref_masked,ref_unmasked,atol=1e-8,rtol=1e-8), (
    "Mask switch does not affect refiner output on a solid-crossing probe."
)

# Check masks for an artificially located solid crossing on a Cartesian stencil.
# This tests the validity-rule difference directly, independently of learned weights.
synthetic_fluid = torch.ones(1,1,9,9)
synthetic_fluid[...,4,4] = 0.0
at_y, at_x = 4, 2  # A +3-x stencil from (4,2) crosses solid (4,4).
original_valid = bool((synthetic_fluid[0,0,at_y,at_x+1:at_x+4] > .999).all())
new_valid = (at_x + 3 < 9)
assert not original_valid and new_valid
out_of_domain_valid = (at_x - 3 >= 0)
assert not out_of_domain_valid

# Validate finite gradients in both models with the unchanged supervised training objective.
for name, model in [("masked", masked_model), ("no_path_rejection", unmasked_model)]:
    model.train(); model.zero_grad(set_to_none=True)
    prediction = model(batch)
    loss, terms = core.training_loss(prediction, batch, {})
    assert torch.isfinite(loss), f"Nonfinite loss: {name}"
    loss.backward()
    assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    print(name, "parameters", core.model_size(model)["real_scalar_parameters"],
          "smoke loss", round(float(loss.detach()),6))
    model.zero_grad(set_to_none=True)

# No weights are stored here: training creates fresh, versioned checkpoints.
del original_model, masked_model, unmasked_model, small_set, batch, latent, probe
if torch.cuda.is_available():torch.cuda.empty_cache()
print("All pretraining gates passed: source parity, identical weights, masks, finite gradients.")

## 5. Register the immutable development study plan

Both models differ by one Boolean flag. The new plan stores dataset, split, original-core, versioned patch and runner identities. An existing plan cannot be silently overwritten, and a checkpoint from another model version cannot be resumed.

In [ ]:
PLAN = {
    "schema":"mask_path_ablation_v1",
    "core_fingerprint":ablation_core.implementation_fingerprint(),
    "original_core_fingerprint":original_fp,
    "core_source_sha256":sha256_file(CORE_PATH),
    "model_patch_sha256":sha256_file(VERSION_FILE),
    "runner_sha256":sha256_file(HELPER_PATH),
    "dataset_id":meta["dataset_id"],
    "split_id":split["split_id"],
    "configs":configs,
    "seeds":SEEDS,
    "source_run_ids":{f"no_alignment_seed{seed}":original_identities[seed]["run_id"] for seed in SEEDS},
    "scope":"Post hoc development-validation mask ablation; no official test access",
    "primary_local_metric":"near_pressure_mae",
    "secondary_metrics":["legacy_l1", "legacy_rel_l1_aggregate", "p_rel_l2",
                         "velocity_rel_l2", "near_velocity_rel_l2", "wake_velocity_rel_l2"],
    "comparison":"unmasked minus masked; positive difference favors mask-path rejection",
    "bootstrap_repetitions":BOOTSTRAP_REPETITIONS,
    "bootstrap_seed":BOOTSTRAP_SEED,
}
PLAN["plan_id"] = stage.json_hash(PLAN)
plan_file = STUDY_ROOT / "mask_ablation_plan.json"
if plan_file.exists():
    if stage.read_json(plan_file) != PLAN:
        raise ValueError("New study plan differs from existing plan. Do not overwrite it.")
else:
    stage.write_json(plan_file, PLAN)
print("Plan ID:", PLAN["plan_id"])
print("Locked study plan:", plan_file)
print("All checkpoints and summaries will be written under:", STUDY_ROOT)

## 6. Resumable training, one chunk per execution

This cell **reuses the original validated helper's** `run_next_chunk()` and `train_one_chunk()` functions from Notebooks 03 to 05. Its checkpoint files retain the model, optimizer, RNG states, selected best score, training history, code fingerprint and plan identity. Each chunk ends at a validation boundary. A Colab disconnect does not require restarting training.

With `ACTION="status"`, only the progress table is printed. Set `ACTION="train"` at the top, execute all cells, then repeat until all six runs have reached 10,000 steps. Do **not** change scientific settings in the same run directory.

In [ ]:
progress_before = stage.progress_table(STUDY_ROOT, PLAN)
display(progress_before)

if ACTION == "train":
    if not torch.cuda.is_available():
        raise RuntimeError("Training requires a Colab GPU runtime.")
    progress_after = stage.run_next_chunk(
        ablation_core, PREPARED_ROOT, split, STUDY_ROOT, PLAN,
        active_names=VARIANTS, chunk_steps=CHUNK_STEPS,
        session_minutes=SESSION_MINUTES
    )
    display(progress_after)
else:
    print("No new optimizer steps executed. Set GCNO_MASK_ACTION=train or ACTION='train'.")

## 7. Audit six completed runs and compute paired validation statistics

The audit checks sample identity, split identity, training budgets, seed pairing, selected checkpoints, finite predictions, and model code provenance. It will not combine partial experiments with completed ones. Positive differences in the main contrast (`no_path_rejection` minus `masked`) favor masking.

The crossed bootstrap treats airfoil geometries, not individual operating conditions, as the statistical grouping unit. With only three training seeds and development identities overlapping training geometries, interpret these intervals as exploratory development evidence, not an independent proof of generalization.

In [ ]:
progress = stage.progress_table(STUDY_ROOT, PLAN)
complete = bool(len(progress)==6 and (progress.status == "completed").all())
if not complete:
    print("Study incomplete; finish all six runs before statistical comparisons.")
    display(progress)
else:
    audit = stage.audit_study(
        PREPARED_ROOT, SPLIT_PATH, STUDY_ROOT,
        names=VARIANTS, seeds=SEEDS, expected_steps=MAX_STEPS,
        expected_configs=PLAN["configs"]
    )
    assert audit["core_fingerprint"] == PLAN["core_fingerprint"]
    mean_sd = audit["runs"].groupby("variant").agg(
        {col:["mean","std"] for col in ["validation_L1","validation_relative_L1",
                                      "validation_pressure_L2","validation_velocity_L2"]}
    )
    display(mean_sd)
    intervals, regional_summary = stage.regional_analysis(
        audit["cases"],
        comparisons=[("no_path_rejection", "masked")],
        repetitions=BOOTSTRAP_REPETITIONS
    )
    stage.write_csv(REPORT_ROOT / "mask_ablation_all_runs.csv", audit["runs"])
    stage.write_csv(REPORT_ROOT / "mask_ablation_case_metrics.csv", audit["cases"])
    stage.write_csv(REPORT_ROOT / "mask_ablation_paired_intervals.csv", intervals)
    stage.write_csv(REPORT_ROOT / "mask_ablation_regional_summary.csv", regional_summary)
    pd.DataFrame(audit["provenance"]).to_csv(REPORT_ROOT / "input_provenance.csv",index=False)
    display(intervals[["metric", "status", "reference_mean", "candidate_mean",
                       "absolute_reduction", "reduction_percent",
                       "absolute_ci_low", "absolute_ci_high"]].round(6))
    primary_row = intervals[intervals.metric == PLAN["primary_local_metric"]]
    print("Primary local diagnostic:")
    display(primary_row.T)

## 8. Publication figures: paired validation comparison

These plots are generated only from completed model predictions and retain paired-seed information. Values are measured results; the notebook does not contain fabricated scores.

In [ ]:
if complete:
    chosen_metrics = ["near_pressure_mae", "near_velocity_rel_l2", "legacy_l1",
                      "p_rel_l2", "velocity_rel_l2", "wake_velocity_rel_l2"]
    subset = intervals[(intervals.metric.isin(chosen_metrics)) & (intervals.status=="ok")].copy()
    fig, axes = plt.subplots(2,3,figsize=(13,7),constrained_layout=True)
    for ax, metric in zip(axes.ravel(), chosen_metrics):
        frame = subset[subset.metric==metric]
        ax.axvline(0, color="0.3", linestyle=":", linewidth=1)
        if frame.empty:
            ax.set_axis_off();continue
        rec = frame.iloc[0]
        ax.plot([rec.absolute_ci_low, rec.absolute_ci_high], [0,0], lw=3)
        ax.scatter([rec.absolute_reduction],[0],s=45,zorder=3)
        ax.set_yticks([])
        ax.set_xlabel("Unmasked error minus masked error")
        ax.set_title(metric,loc="left")
        ax.ticklabel_format(axis="x",style="sci",scilimits=(-3,3))
        ax.grid(axis="x",alpha=.15)
    fig.suptitle("Development validation: paired 95% crossed-bootstrap intervals")
    fig.savefig(REPORT_ROOT/"mask_path_paired_intervals.png",dpi=200)
    plt.show();plt.close(fig)

    key = "near_pressure_mae"
    byseed = audit["cases"].groupby(["variant","seed"])[key].mean().unstack(0)
    fig,ax=plt.subplots(figsize=(6,4))
    for seed,row in byseed.iterrows():
        ax.plot(["Unmasked","Masked"],[row["no_path_rejection"],row["masked"]],marker="o",label=f"seed {seed}")
    ax.set_ylabel(key)
    ax.set_title("Matched seed effects, near-airfoil pressure MAE")
    ax.legend();ax.grid(alpha=.15)
    fig.tight_layout();fig.savefig(REPORT_ROOT/"mask_path_seed_effects.png",dpi=200)
    plt.show();plt.close(fig)

## 9. Optional matched CFD figures, development validation only

The visualization selects three cases by **unmasked-control error** (median, 95th percentile, and worst at seed 42). This selection is diagnostic, not representative. Identical reference/prediction limits are used within each variable; absolute-error limits are shared between methods. Nothing from the official test split is accessed.

In [ ]:
if complete and MAKE_FIGURES:
    seed = SEEDS[0]
    failure_order = audit["cases"].query("variant == 'no_path_rejection' and seed == @seed")
    failure_order = failure_order.sort_values(["legacy_rel_l1_aggregate","sample_id"]).reset_index(drop=True)
    indices = sorted(set(int(round((len(failure_order)-1)*q)) for q in [0.50,0.95,1.0]))
    chosen = failure_order.iloc[indices][["sample_id","airfoil_id","legacy_rel_l1_aggregate"]]
    chosen.to_csv(REPORT_ROOT/"chosen_development_cases.csv",index=False)

    selected_ids = [str(x) for x in chosen.sample_id]
    validation_manifest = audit["validation_manifest"]
    validation_ids = list(validation_manifest.index.astype(str))
    val_ds = core.AirfoilDataset(PREPARED_ROOT,"train",split["val_indices"],stats, configs["masked"])
    sample_map = {sid: val_ds[validation_ids.index(sid)] for sid in selected_ids}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models={}
    for name in VARIANTS:
        folder = STUDY_ROOT / f"{name}_seed{seed}"
        identity = audit["identities"][(name,seed)]
        checkpoint = torch.load(folder/"best.pt",map_location="cpu",weights_only=False)
        if checkpoint["run_id"] != identity["run_id"]:
            raise ValueError(f"Checkpoint identity mismatch: {name}")
        model = ablation_core.model_factory(identity["config"]).to(device)
        model.load_state_dict(checkpoint["model"],strict=True)
        model.eval();models[name]=model
    try:
        for sid,item in sample_map.items():
            b = {k:v.unsqueeze(0) if torch.is_tensor(v) else [v] for k,v in item.items()}
            b = core.to_device(b,device)
            with torch.inference_mode():
                truth=core.common_gauge(core.legacy_to_star(b["target"]),b["fluid"])[0].cpu().numpy()
                predictions={name:core.common_gauge(core.legacy_to_star(m(b)["prediction"]),b["fluid"])[0].cpu().numpy()
                             for name,m in models.items()}
            fluid=b["fluid"][0,0].cpu().numpy()>0.5
            labels=["u / U","v / U","kinematic pressure / U²"]
            fig,axes=plt.subplots(3,5,figsize=(17,10),constrained_layout=True)
            for ch,label in enumerate(labels):
                fields=[truth[ch],predictions["no_path_rejection"][ch],predictions["masked"][ch]]
                lo=min(float(np.min(f[fluid])) for f in fields)
                hi=max(float(np.max(f[fluid])) for f in fields)
                err_un=np.abs(fields[1]-fields[0]);err_mask=np.abs(fields[2]-fields[0])
                high=max(float(err_un[fluid].max()),float(err_mask[fluid].max()),1e-8)
                for j,fld in enumerate(fields):
                    im=axes[ch,j].imshow(np.ma.masked_where(~fluid,fld),origin="lower",
                        cmap="viridis",vmin=lo,vmax=hi,interpolation="nearest")
                    axes[ch,j].set_title(["CFD","No path rejection","GCNO-LR masked"][j]+": "+label)
                    fig.colorbar(im,ax=axes[ch,j],fraction=0.046)
                for j,err in enumerate([err_un,err_mask],start=3):
                    ie=axes[ch,j].imshow(np.ma.masked_where(~fluid,err),origin="lower",
                        cmap="magma",vmin=0,vmax=high,interpolation="nearest")
                    axes[ch,j].set_title(["","","","Unmasked error","Masked error"][j]+": "+label)
                    fig.colorbar(ie,ax=axes[ch,j],fraction=0.046)
                for ax in axes[ch]:ax.set_xticks([]);ax.set_yticks([])
            fig.suptitle(f"Development validation {sid}, seed {seed}")
            fig.savefig(REPORT_ROOT/f"validation_compare_{sid.replace('/','_')}.png",dpi=160)
            plt.show();plt.close(fig)
    finally:
        del models, val_ds
        if torch.cuda.is_available():torch.cuda.empty_cache()

## 10. Optional matched inference-cost check

Latency is not the primary objective. Mask-aware validity checks have additional `grid_sample` calls, so this optional test measures whether removing them saves inference time. Measurements use resident inputs in a common GPU session and exclude preprocessing, transfers, and file I/O. They do not measure training cost.

In [ ]:
if complete and RUN_INFERENCE_TIMING:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable GPU for the matched inference test.")
    device=torch.device("cuda")
    val_ds=core.AirfoilDataset(PREPARED_ROOT,"train",split["val_indices"],stats,configs["masked"])
    timing=[]
    for name in VARIANTS:
        ckpt = torch.load(STUDY_ROOT/f"{name}_seed{SEEDS[0]}"/"best.pt",
                          map_location="cpu",weights_only=False)
        model=ablation_core.model_factory(configs[name]).to(device)
        model.load_state_dict(ckpt["model"]);model.eval()
        for bs in [1,10]:
            dl=core.DataLoader(val_ds,batch_size=bs,shuffle=False,num_workers=0)
            batch=next(iter(dl))
            metrics=core.benchmark_inference(model,batch,device,repeats=50,warmup=10)
            timing.append({"variant":name,"seed":SEEDS[0],**metrics})
            del dl,batch
        del model,ckpt
    timing=pd.DataFrame(timing)
    stage.write_csv(REPORT_ROOT/"mask_ablation_inference_cost.csv",timing)
    display(timing)

## 11. Export review-ready results and manuscript interpretation

The added experiment can support a **mask-specific** claim only if the controlled comparison favors masking. If the interval overlaps zero, or the unmasked variant improves, acknowledge that the broader local refinement branch is helpful without establishing that path rejection is independently beneficial. Do not describe post hoc development analysis as a prespecified official-test comparison.

In [ ]:
if complete:
    result_rows=[]
    for name,frame in audit["runs"].groupby("variant"):
        row={"variant":name,"n_seeds":len(frame),"parameters":int(frame.parameters.iloc[0])}
        for key in ["validation_L1","validation_relative_L1","validation_pressure_L2","validation_velocity_L2"]:
            row[key+"_mean"]=float(frame[key].mean())
            row[key+"_sd"]=float(frame[key].std(ddof=1))
        result_rows.append(row)
    manuscript_table=pd.DataFrame(result_rows)
    stage.write_csv(REPORT_ROOT/"manuscript_mask_ablation_table.csv",manuscript_table)
    focus=intervals[intervals.metric==PLAN["primary_local_metric"]].iloc[0]
    finding=(
      "The masked variant has an exploratory positive paired interval on the declared local metric."
      if focus.status=="ok" and focus.absolute_ci_low>0 else
      "The exploratory paired interval does not establish a positive local-metric effect of masking."
    )
    summary={"post_hoc":True,"official_test_used":False,"plan_id":PLAN["plan_id"],
             "model_patch_sha256":PLAN["model_patch_sha256"],
             "primary_local_metric":PLAN["primary_local_metric"],
             "conclusion":finding,
             "limitations":["Development validation includes overlapping airfoil identities with training",
                            "Three random seeds only",
                            "No new independent test", "No assertion of conservation enforcement"]}
    stage.write_json(REPORT_ROOT/"manuscript_interpretation.json",summary)
    stage.write_json(REPORT_ROOT/"output_manifest.json",{
        "plan_id":PLAN["plan_id"],
        "generated_utc":datetime.now(timezone.utc).isoformat(),
        "files":{p.name:sha256_file(p) for p in REPORT_ROOT.iterdir() if p.is_file() and p.name!='output_manifest.json'}
    })
    print(finding)
    display(manuscript_table)
    print("Final validated outputs:", REPORT_ROOT)
else:
    print("Run all six configurations first. No metrics or conclusions are invented.")

## Scientific reporting checklist

- **Experimental status:** Added post hoc ablation on the original development split only; original frozen test is unchanged.
- **Controlled factor:** solid-path rejection versus in-domain sampling through solid, with all other model and training choices matched.
- **Primary diagnostic:** `near_pressure_mae`; report global field, velocity, and wake metrics as contextual checks.
- **Statistical unit:** airfoil ID within paired seed; do not treat operating conditions from the same airfoil as independent identities.
- **Limitations:** training/validation may share identities; only three seeds; independent holdout not repeated; no physical conservation inference.
- **Manuscript editing:** add a local-control row to the development ablation table, paired CI results, representative error figures, and a modest conclusion whether mask validity is independently beneficial.

**Do not use the existing official-test outputs to optimize this control.** If further independent confirmation is desired, define a genuinely new geometry-disjoint validation/test set or external benchmark before model selection.